# Ablation Study — Results
Loads from `checkpoints/ablation_results.pkl`. No training happens here.

In [ ]:
import sys
sys.path.insert(0, '../src')

from config import ABLATION_RESULTS
from train import load_results
from plot import plot_ablation_bar, plot_multi_history
from ablation import print_results_table, print_tta_table
import matplotlib.pyplot as plt

## Load saved results

In [ ]:
results = load_results(ABLATION_RESULTS)
arch_results = results.get('arch', [])
reg_results  = results.get('reg',  [])
tta_results  = results.get('tta',  {})
print(f'Arch experiments: {len(arch_results)}')
print(f'Reg experiments:  {len(reg_results)}')
print(f'TTA strategies:   {len(tta_results)}')

## Architecture ablations

In [ ]:
print_results_table(arch_results, 'Architecture Ablations')
plot_ablation_bar(arch_results, title='Architecture Ablations',
                  save_name='arch_ablation_bar.png')
plot_multi_history(arch_results, title='Validation Accuracy — Architecture Variants',
                   save_name='arch_ablation_curves.png')

## Regularization ablations

In [ ]:
print_results_table(reg_results, 'Regularization Ablations')
plot_ablation_bar(reg_results, title='Regularization Ablations',
                  save_name='reg_ablation_bar.png')
plot_multi_history(reg_results, title='Validation Accuracy — Regularization Variants',
                   save_name='reg_ablation_curves.png')

## TTA results

In [ ]:
if tta_results:
    print_tta_table(tta_results)

    # Bar chart
    keys = list(tta_results.keys())
    vals = [tta_results[k] for k in keys]
    order = sorted(range(len(vals)), key=lambda i: -vals[i])

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh([keys[i] for i in order], [vals[i] for i in order])
    ax.bar_label(bars, fmt='{:.4f}', padding=3)
    ax.set_xlabel('Accuracy')
    ax.set_title('Inference-Time Aggregation Strategies')
    plt.tight_layout()
    plt.savefig('../figures/tta_results.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No TTA results found. Run: python scripts/run_ablation.py --study tta')

## Key findings summary

In [ ]:
if arch_results:
    best_arch = max(arch_results, key=lambda r: r['best_val_acc'])
    worst_arch = min(arch_results, key=lambda r: r['best_val_acc'])
    print(f"Best architecture:  {best_arch['exp_name']}  ({best_arch['best_val_acc']:.4f})")
    print(f"Worst architecture: {worst_arch['exp_name']} ({worst_arch['best_val_acc']:.4f})")

if reg_results:
    best_reg = max(reg_results, key=lambda r: r['best_val_acc'])
    print(f"Best regularization: {best_reg['exp_name']} ({best_reg['best_val_acc']:.4f})")
    print(f"  Config: {best_reg['config']}")

if tta_results:
    best_tta = max(tta_results, key=tta_results.get)
    print(f"Best TTA strategy: {best_tta} ({tta_results[best_tta]:.4f})")